# FT-900 训练起点前置原件

按冻结计划读取 2010-10 至 2010-12 原始月报，只补来源，不计算收益、因子或账户。

In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
import qlib,pandas as pd
from qlib.data import D
root=Path.cwd();card=root/'research/census_ft900_train_leadin_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(root/'research/census_ft900_semiconductor_export_source_20261023/full_training_row_audit.json')==plan['parent_training_row_audit_sha256']
index=root/plan['official_index']['path'];assert sha(index)==plan['official_index']['sha256'] and index.stat().st_size==plan['official_index']['bytes']
soup=BeautifulSoup(index.read_bytes(),'html.parser');found={}
for div in soup.select('div.uscb-list-item-container'):
 p=div.find('p');title=' '.join(p.get_text(' ',strip=True).split()) if p else ''
 if title in ('October 2010','November 2010','December 2010'):
  urls={a['href'] for a in div.select('a[href]') if re.search(r'/ft900_\d{4}\.pdf$',a['href'])}
  assert len(urls)==1
  found[title]=next(iter(urls))
assert len(found)==3 and sorted(found.values())==sorted(plan['fixed_urls'])
provider=root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'
qlib.init(provider_uri=str(provider),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2011-03-10',freq='day'))
assert len(calendar)==47 and str(calendar[0].date())=='2011-01-03' and str(calendar[-1].date())=='2011-03-10'
snapshot=root/'data/snapshots/census_ft900_train_leadin_20261023';snapshot.mkdir(parents=True,exist_ok=True)
print({'official_links':found,'early_Stage1_sessions_before_2011_reference_release':len(calendar)})


[36771:MainThread](2026-10-23 00:34:06,564) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[36771:MainThread](2026-10-23 00:34:07,138) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[36771:MainThread](2026-10-23 00:34:07,139) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'official_links': {'December 2010': 'https://www.census.gov/foreign-trade/Press-Release/ft900/ft900_1012.pdf', 'November 2010': 'https://www.census.gov/foreign-trade/Press-Release/ft900/ft900_1011.pdf', 'October 2010': 'https://www.census.gov/foreign-trade/Press-Release/ft900/ft900_1010.pdf'}, 'early_Stage1_sessions_before_2011_reference_release': 47}


In [2]:
from datetime import datetime,timezone
from io import BytesIO
import requests
from pypdf import PdfReader
receipts=[];rows=[]
for month,url in zip(plan['fixed_reference_months'],plan['fixed_urls']):
 name=url.rsplit('/',1)[-1];dest=snapshot/name
 if dest.exists():raise ValueError('fixed source GET already consumed; do not silently retry')
 start=datetime.now(timezone.utc).isoformat();response=requests.get(url,timeout=40)
 end=datetime.now(timezone.utc).isoformat();assert response.status_code==200 and response.headers.get('Content-Type','').startswith('application/pdf')
 body=response.content;assert body.startswith(b'%PDF-') and len(body)<4_000_000
 dest.write_bytes(body)
 texts=[page.extract_text() or '' for page in PdfReader(BytesIO(body)).pages]
 first=texts[0];assert f'FT-900 ({month[2:4]}-{month[5:7]})' in first
 full_month=pd.Timestamp(month+'-01').strftime('%B %Y');assert full_month.lower() in first.lower()
 header=re.search(r'FOR IMMEDIATE RELEASE[\s\S]{0,100}?\b(?:MONDAY|TUESDAY|WEDNESDAY|THURSDAY|FRIDAY)\b[^\n]*',first,re.I)
 if not header:header=re.search(r'FOR RELEASE AT[^\n]*',first,re.I)
 assert header,(month,first[:500])
 candidates=[]
 for page_no,text in enumerate(texts,1):
  if 'Exhibit 7.' not in text or 'Exports of Goods by End-Use Category and Commodity' not in text:continue
  match=re.search(r'Semiconductors\s+([\d,]+)\s+([\d,]+)\s+(-?[\d,]+)',text)
  if match:candidates.append((page_no,tuple(int(x.replace(',','')) for x in match.groups())))
 assert len(candidates)==1,(month,candidates)
 row={'reference_month':month,'source_sha256':sha(dest),'source_bytes':len(body),'printed_month':full_month,
      'release_header':header.group(0),'Exhibit7_pdf_page':candidates[0][0],
      'current_month_million_usd':candidates[0][1][0],'prior_month_same_release_million_usd':candidates[0][1][1],
      'printed_change_million_usd':candidates[0][1][2]}
 rows.append(row)
 receipts.append({'url':url,'requested_at_utc':start,'received_at_utc':end,'status':response.status_code,
  'Date':response.headers.get('Date'),'Last-Modified':response.headers.get('Last-Modified'),
  'Content-Type':response.headers.get('Content-Type'),'bytes':len(body),'sha256':sha(dest)})
result={'status':'three_original_2010_lead_in_rows_verified_source_only','plan_sha256':sha(card/'plan.json'),
 'source_index_sha256':plan['official_index']['sha256'],'early_Stage1_Qlib_sessions':len(calendar),
 'rows':rows,'receipts':receipts,'historical_first_public_bytes_qualified':False,'full_calendar_availability_qualified':False,
 'market_outcome_reads':0,'factor_values':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'rows':[(x['reference_month'],x['current_month_million_usd']) for x in rows]})


{'status': 'three_original_2010_lead_in_rows_verified_source_only', 'rows': [('2010-10', 3592), ('2010-11', 3808), ('2010-12', 3860)]}


In [3]:
import tarfile,tempfile,shlex,subprocess
from src.data.nas_market_data import _settings,nas_api_call,_ssh
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='ft900-leadin-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for file in sorted(snapshot.iterdir()):
   if file.is_file() and file.name!='.gitignore':tar.add(file,arcname='source/'+file.name)
  tar.add(index,arcname='source/ft900_index.html')
 digest=sha(archive)
 receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
   path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='ft900-leadin-restore-') as restored:
  copy=Path(restored)/'source.tar.gz'
  with copy.open('wb') as out:
   proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+remote],stdout=out,stderr=subprocess.PIPE,timeout=60)
  assert proc.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as source:
   restored_hash={}
   for member in source.getmembers():
    assert member.isfile() and member.name.startswith('source/')
    restored_hash[member.name.split('/')[-1]]=hashlib.sha256(source.extractfile(member).read()).hexdigest()
 expected={p.name:sha(p) for p in snapshot.iterdir() if p.is_file() and p.name!='.gitignore'};expected['ft900_index.html']=sha(index)
 assert restored_hash==expected
 result.update(nas_archive_sha256=digest,nas_remote_hash_verified=True,nas_scoped_restore_verified=True)
 (card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
 (card/'nas_binding.json').write_text(json.dumps({'status':'three_2010_original_PDFs_and_index_NAS_restored','archive_sha256':digest,
  'archive_bytes':receipt['size_bytes'],'source_file_sha256':expected,'nas_artifact':receipt,
  'remote_hash_verified':True,'scoped_restore_verified':True},ensure_ascii=False,indent=2)+'\n')
 summary={'experiment':'census_ft900_train_leadin_20261023','decision':'three_prior_original_months_fill_source_lead_in_no_factor',
  'objective_achieved':False,'early_Stage1_Qlib_sessions':len(calendar),'source_rows_verified':3,
  'result':'result.json','nas_binding':'nas_binding.json','historical_first_public_bytes_qualified':False,
  'full_calendar_availability_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
  'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[],
   'raw_source_nas_archive_sha256':digest}}
 (card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
 print({'NAS':digest,'restored_members':len(expected)})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'NAS': '5d92aba136b0462dab9994bef767398f778848664b044d2c5e7eb5e52facd87f', 'restored_members': 4}


## 中文结论

原 Stage1 训练起点至 2011-01 参考月 FT-900 发布日共有 47 个 Qlib 会话。已核对 2010-10、11、12 三份原始报告的月份、发行声明及 Exhibit 7 半导体行，原始字节与官方索引已存 NAS 并做远端哈希及范围恢复（`5d92aba136b0462dab9994bef767398f778848664b044d2c5e7eb5e52facd87f`）。2010-11 当月值 3808 与后续 2010-12 报告中的上月值 3804 不同，后续只能用同报告环比。

零市场收益读取、零因子、零拟合、零账户。三份前置月报提供来源候选，不证明历史首次公开字节或完整训练/诊断会话对齐，不能宣称 SOXX 超额。详见 `result.json`、`summary.json`、`nas_binding.json` 和 `conclusion.md`。
